# MEVID-SameClothes: a standalone dataset folder

Creates a new dataset next to the original MEVID on your Drive: **`Data/MEVID-SameClothes/`**, in which **every person appears in exactly one outfit** (train and test). It has the same layout and file formats as MEVID, so the same code can read it.

**Rule (labels only, fixed in advance):** for each person, keep the outfit seen on the most cameras (ties: most clips, then lowest outfit id).

**Contents when finished**
- `mevid-sc-bbox-train.tar`, `mevid-sc-bbox-test.tar`: only the kept images, as `bbox_<split>/<pid>/<image>.jpg`
- `annotations/`: `train_name.txt`, `test_name.txt`, `track_train_info.txt`, `track_test_info.txt`, `query_IDX.txt` (MEVID format), plus the outfit choice per person and a map back to the original MEVID rows
- `SUBSET_REPORT.json` (counts and verification) and `README.txt`

The original MEVID files are not changed. Normal runtime, no GPU. Total time about 30 to 45 minutes, mostly reading the original train archive.

In [ ]:
# 1. Mount Drive, find MEVID, create the new dataset folder
import os, glob, json, re, shutil, subprocess, tarfile, numpy as np, pandas as pd
from datetime import date
from google.colab import drive
drive.mount('/content/drive')

src = (glob.glob('/content/drive/Shareddrives/*/Data/MEVID/mevid-v1-bbox-test.tgz')
       + glob.glob('/content/drive/MyDrive/**/mevid-v1-bbox-test.tgz',
                   recursive=True))
assert src, 'MEVID archives not found on Drive'
MEVID_DIR = os.path.dirname(src[0])
SC_DIR = os.path.join(os.path.dirname(MEVID_DIR), 'MEVID-SameClothes')
os.makedirs(f'{SC_DIR}/annotations', exist_ok=True)
print('Original MEVID :', MEVID_DIR)
print('New dataset    :', SC_DIR)

subprocess.run(['unzip', '-oq', f'{MEVID_DIR}/mevid-v1-annotation-data.zip',
                '-d', '/content/mevid_ann'], check=True)
def ann(name):
    return glob.glob(f'/content/mevid_ann/**/{name}', recursive=True)[0]


In [ ]:
# 2. Select ONE outfit per person (labels only) and build the new label files
PAT = re.compile(r'(\d{4})O(\d{3})C(\d+)T(\d{3})F(\d+)\.jpg$')

def load(split):
    names = [l.strip() for l in open(ann(f'{split}_name.txt')) if l.strip()]
    info = np.loadtxt(ann(f'track_{split}_info.txt')).astype(np.int64)
    return names, info            # MEVID: 0-based, end inclusive

def choose_outfits(info):
    df = pd.DataFrame(info[:, 2:5], columns=['pid', 'outfit', 'cam'])
    g = df.groupby(['pid', 'outfit']).agg(cameras=('cam', 'nunique'),
                                         clips=('cam', 'size')).reset_index()
    best = (g.sort_values(['pid', 'cameras', 'clips', 'outfit'],
                          ascending=[True, False, False, True])
            .groupby('pid').head(1))
    g['chosen'] = g.set_index(['pid', 'outfit']).index.isin(
        best.set_index(['pid', 'outfit']).index)
    return dict(zip(best.pid, best.outfit)), g

built = {}
for split in ['train', 'test']:
    names, info = load(split)
    chosen, table = choose_outfits(info)
    keep = np.array([info[i, 3] == chosen[info[i, 2]] for i in range(len(info))])
    # skip MEVID's one malformed train row (start/end outside its own clip)
    ok = []
    for i in np.flatnonzero(keep):
        s, e, p, o, c = info[i]
        fr = names[s:e + 1]
        m0, m1 = PAT.search(fr[0]) if fr else None, PAT.search(fr[-1]) if fr else None
        good = (m0 and m1 and int(m0.group(1)) == p and int(m1.group(1)) == p
                and int(m0.group(2)) == o and int(m1.group(2)) == o)
        ok.append(bool(good))
    keep_idx = np.flatnonzero(keep)[np.array(ok)]
    new_names, new_info = [], []
    for i in keep_idx:
        s, e, p, o, c = info[i]
        start = len(new_names)
        new_names += names[s:e + 1]
        new_info.append([start, len(new_names) - 1, p, o, c])
    new_info = np.array(new_info, dtype=np.int64)
    built[split] = dict(names=new_names, info=new_info, orig_idx=keep_idx,
                        table=table)
    table.to_csv(f'{SC_DIR}/annotations/outfit_choice_{split}.csv', index=False)
    print(f'{split}: kept {len(keep_idx):,} of {len(info):,} clips, '
          f'{len(new_names):,} images, {len(chosen)} people')

# queries: official queries in the kept test clips that still have a
# same-person clip on ANOTHER camera among the kept non-query clips
qidx = np.loadtxt(ann('query_IDX.txt')).astype(np.int64)
t = built['test']; pos = {o: n for n, o in enumerate(t['orig_idx'])}
kq = [q for q in qidx if q in pos]
non_q = np.array([n for o, n in pos.items() if o not in set(qidx)])
pid_n, cam_n = t['info'][:, 2], t['info'][:, 4]
new_q = [pos[q] for q in kq
         if ((pid_n[non_q] == pid_n[pos[q]]) & (cam_n[non_q] != cam_n[pos[q]])).any()]
t['query'] = np.array(new_q, dtype=np.int64)

for split in ['train', 'test']:
    b = built[split]
    with open(f'{SC_DIR}/annotations/{split}_name.txt', 'w') as f:
        f.write('\n'.join(b['names']) + '\n')
    np.savetxt(f'{SC_DIR}/annotations/track_{split}_info.txt', b['info'], fmt='%d')
    np.savetxt(f'{SC_DIR}/annotations/original_index_{split}.txt',
               b['orig_idx'], fmt='%d')
np.savetxt(f'{SC_DIR}/annotations/query_IDX.txt', t['query'], fmt='%d')
print(f'test queries kept: {len(t["query"])} of {len(qidx)}')


In [ ]:
# 3. Copy ONLY the kept images into new archives (streams the originals;
#    nothing is unpacked on Drive). Train ~20-30 min, test ~10 min.
LOCAL = '/content/sc_build'
os.makedirs(LOCAL, exist_ok=True)
for split in ['test', 'train']:
    out = f'{SC_DIR}/mevid-sc-bbox-{split}.tar'
    if os.path.exists(out):
        print(f'{split}: archive already on Drive, skipping'); continue
    wanted = set(built[split]['names'])
    tmp = f'{LOCAL}/mevid-sc-bbox-{split}.tar'
    n = 0
    with tarfile.open(f'{MEVID_DIR}/mevid-v1-bbox-{split}.tgz', 'r|gz') as src_t, \
         tarfile.open(tmp, 'w') as dst_t:
        for m in src_t:
            if m.isfile() and os.path.basename(m.name) in wanted:
                m.name = f'bbox_{split}/' + os.path.basename(m.name)[:4] + '/' \
                         + os.path.basename(m.name)
                dst_t.addfile(m, src_t.extractfile(m))
                n += 1
                if n % 200000 == 0:
                    print(f'  {split}: {n:,} images copied')
    print(f'{split}: {n:,} images (expected {len(wanted):,}); copying to Drive')
    shutil.copy(tmp, out); os.remove(tmp)


In [ ]:
# 4. VERIFY the new dataset and write its README + report
report = {'built_on': str(date.today()),
          'source': 'MEVID v1 (Davila et al., WACV 2023), CC-BY-4.0',
          'rule': 'per person keep the outfit seen on the most cameras '
                  '(ties: most clips, then lowest outfit id); labels only'}
for split in ['train', 'test']:
    b = built[split]
    names_in_tar = {}
    with tarfile.open(f'{SC_DIR}/mevid-sc-bbox-{split}.tar') as tf:
        for m in tf:
            if m.isfile():
                names_in_tar[os.path.basename(m.name)] = 1
    outfits = {}
    for nm in b['names']:
        mm = PAT.search(nm)
        outfits.setdefault(mm.group(1), set()).add(mm.group(2))
    one_outfit = all(len(v) == 1 for v in outfits.values())
    images_ok = set(b['names']) <= set(names_in_tar)
    assert one_outfit, f'{split}: a person has more than one outfit'
    assert images_ok, f'{split}: some listed images are missing from the archive'
    report[split] = {'people': len(outfits), 'clips': int(len(b['info'])),
                     'images': len(b['names']),
                     'cameras': int(len(np.unique(b['info'][:, 4]))),
                     'max outfits per person': 1}
report['test']['queries'] = int(len(built['test']['query']))
report['test']['gallery'] = int(len(built['test']['info']) - len(built['test']['query']))
report['verification'] = ('PASSED: every person has exactly one outfit in both '
                          'splits; every listed image is in the archive; every '
                          'query has a same-outfit match on another camera')
json.dump(report, open(f'{SC_DIR}/SUBSET_REPORT.json', 'w'), indent=2)
open(f'{SC_DIR}/README.txt', 'w').write(
    'MEVID-SameClothes: MEVID v1 restricted to ONE outfit per person.\n'
    'Same layout as MEVID: annotations/{train,test}_name.txt, '
    'track_{train,test}_info.txt (start end pid outfit cam, 0-based, end '
    'inclusive), query_IDX.txt (0-based).\n'
    'Images: mevid-sc-bbox-{train,test}.tar -> bbox_{split}/<pid>/<image>.jpg\n'
    'original_index_*.txt maps each clip to its row in the original MEVID files.\n'
    'Selection rule and verification: SUBSET_REPORT.json.\n')
print(json.dumps(report, indent=2))
print('\nDataset folder:', SC_DIR, '\n', sorted(os.listdir(SC_DIR)))


In [ ]:
# 5. DATASET STATISTICS: size, people, and cameras per person
#    (MEVID-SameClothes vs original MEVID), saved for the paper.
def size_gb(path):
    return round(os.path.getsize(path) / 1e9, 2) if os.path.exists(path) else 'pending'

def stats(info, n_images, size):
    df = pd.DataFrame(info[:, 2:5], columns=['pid', 'outfit', 'cam'])
    df['frames'] = info[:, 1] - info[:, 0] + 1
    cams_pp = df.groupby('pid').cam.nunique()
    return {
        'people': int(df.pid.nunique()),
        'outfits': int(df[['pid', 'outfit']].drop_duplicates().shape[0]),
        'clips (tracklets)': int(len(df)),
        'images (frames)': int(n_images),
        'size on disk (GB)': size,
        'cameras in split': int(df.cam.nunique()),
        'cameras per person: mean': round(float(cams_pp.mean()), 2),
        'cameras per person: median': float(cams_pp.median()),
        'cameras per person: min-max': f'{cams_pp.min()}-{cams_pp.max()}',
        'people on >=2 cameras': int((cams_pp >= 2).sum()),
        'people on >=5 cameras': int((cams_pp >= 5).sum()),
        'clips per person: mean': round(float(df.groupby('pid').size().mean()), 1),
        'frames per clip: mean': round(float(df.frames.mean()), 1),
    }

rows, per_person = {}, []
for split in ['train', 'test']:
    b = built[split]
    rows[f'SameClothes {split}'] = stats(
        b['info'], len(b['names']), size_gb(f'{SC_DIR}/mevid-sc-bbox-{split}.tar'))
    names, info = load(split)
    rows[f'Original {split}'] = stats(
        info, len(names), size_gb(f'{MEVID_DIR}/mevid-v1-bbox-{split}.tgz'))
    df = pd.DataFrame(b['info'][:, 2:5], columns=['pid', 'outfit', 'cam'])
    for p, g in df.groupby('pid'):
        per_person.append({'split': split, 'pid': int(p),
                           'outfit kept': int(g.outfit.iloc[0]),
                           'cameras': int(g.cam.nunique()),
                           'camera ids': ' '.join(map(str, sorted(g.cam.unique()))),
                           'clips': int(len(g))})
rows['SameClothes test']['queries'] = int(len(built['test']['query']))
rows['SameClothes test']['gallery clips'] = int(
    len(built['test']['info']) - len(built['test']['query']))

table = pd.DataFrame(rows)[['SameClothes train', 'SameClothes test',
                            'Original train', 'Original test']]
table.to_csv(f'{SC_DIR}/DATASET_STATISTICS.csv')
pp = pd.DataFrame(per_person)
pp.to_csv(f'{SC_DIR}/cameras_per_person.csv', index=False)
json.dump(rows, open(f'{SC_DIR}/DATASET_STATISTICS.json', 'w'), indent=2)

print('DATASET STATISTICS')
display(table)
print('\nHow many people appear on N cameras (SameClothes):')
display(pp.pivot_table(index='cameras', columns='split', values='pid',
                       aggfunc='count', fill_value=0))
print('Saved DATASET_STATISTICS.csv/.json and cameras_per_person.csv to', SC_DIR)